# Get Information About Schools (GIAS)

This notebook combines data downloaded from GIAS (https://get-information-schools.service.gov.uk/Downloads).

Further information can be found in the README.md in the edubase data folder

In [2]:
# Import libraries
import numpy as np
import pyodbc
import pandas as pd
import logging
import datetime
from datetime import datetime
import glob
from pathlib import Path
import os
import re
import random
from utils.connection_utils import connect_to_postgres

# Set display options for the notebook
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 2000)
# pd.options.display.max_columns = None

from warnings import filterwarnings
filterwarnings("ignore", category=UserWarning, message='.*pandas only supports SQLAlchemy connectable.*')

# Clear previous logging configuration and configure a new one
for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)

logging.basicConfig(level=logging.INFO,
                    format='%(asctime)s %(levelname)s %(message)s',
                    handlers=[logging.StreamHandler(), logging.FileHandler("notebook_log.log")])

logger = logging.getLogger(__name__)

In [3]:
# Get current working directory
dir_code = os.getcwd()
print("Current working directory:", dir_code)

# Define directory where data are stored, starting from your home directory
home = Path.home()
dir_data = home / 'OneDrive - Ambition Institute' / 'code' / 'DfE_data' / 'data' / 'get-information-about-schools'
print("Data directory:", dir_data)

# Change directory
if dir_data.exists():
    os.chdir(dir_data)
    print(f"Changed working directory to: {Path.cwd()}")
else:
    print(f"Directory does not exist: {dir_data}")

Current working directory: C:\Users\stefanie.meliss\OneDrive - Ambition Institute\code\DfE_data
Data directory: C:\Users\stefanie.meliss\OneDrive - Ambition Institute\code\DfE_data\data\get-information-about-schools
Changed working directory to: C:\Users\stefanie.meliss\OneDrive - Ambition Institute\code\DfE_data\data\get-information-about-schools


## Yearly School Information

The following variables can be used to identify schools:

**Unique Reference Number (URN)**  
The URN is a sequential and unique number automatically assigned by the DfE’s Get Information about Schools (GIAS) system when a record is created. Children centres are 5 digits and start with a 2. Welsh establishments are 6 digits and start with a 4. All other establishments are 6 digits and start with a 1.
  
**DfE number**  
The DfE number is a concatenation of a **3-digit local authority number** (the local authority that administers the establishment) and a **4-digit establishment number**, usually separated by a slash. This creates a 7-digit number. This number is issued by DfE to all local authority nurseries and compulsory school-aged educational establishments, including further education colleges, that are recorded on Get Information about Schools (GIAS).
  
The DfE number is also known as: 
- The Department for Education number, the Department for Education no, the DfE no;
- the DfE establishment number, the DfE establishment no, the DfE estab number, the DfE estab no;
- the local authority establishment number, the local authority establishment no, the local authority estab number, the local authority estab no;
- the LA establishment number, the LA establishment no, the LA estab number, the LA estab no (however this is one part of the two components that make the DfE number);
- the establishment number, the establishment no, the estab number, the estab no (however this is one part of the two components that make the DfE number).
  
**LAESTAB**  
Local authority establishment. A 3-digit local authority code, followed by a 4-digit establishment code.  

**Local authority number**  
The local authority number is also known as the local authority no, the LA number, the LA no.
The local authority number is a 3 digit number assigned to each local authority.
This number is combined with the 4 digit establishment number to create the 7 digit DfE number


**Establishment number**  
The establishment number is also known as the establishment no, the estab number, the estab no. The establishment number is a 4 digit number.
The number is assigned by the DfE to all local authority nurseries and compulsory school aged educational establishments, including further education colleges, that are recorded on Get Information about Schools (GIAS).
  
**A note on Children’s centres:**  
Children’s centres are included in GIAS for reference and data purposes. However, according to DfE documentation and GIAS, children’s centres do not have an LAESTAB code in the same way as schools and nurseries.
They may have a URN but not always an LAESTAB, because they are not classified as schools or educational establishments for the purposes of DfE funding and reporting.  

For a more detailed explanation on the terms used in the GIAS, please see: https://get-information-schools.service.gov.uk/glossary

In [5]:
# Define the columns to keep and their new names
columns_to_keep = {
        "CensusDate": "Census_Date",
        "URN": "URN", # unique reference number
        "EstablishmentName": "School_Name",
        "LA (code)": "LA_code", # 3-digit LA number
        "EstablishmentNumber": "Estab_Number", # 4-digit estab number
        "LA (name)": "Local_Authority",
        "Postcode": "Postcode",
        "GOR (name)": "Region",
        "UrbanRural (name)": "Urban_Rural",
        "TypeOfEstablishment (name)": "School_Type",
        "EstablishmentTypeGroup (name)": "School_Type_Group",
        "EstablishmentStatus (name)": "School_Status",
        "PhaseOfEducation (name)": "School_Phase",
        "StatutoryLowAge": "School_Age_Low",
        "StatutoryHighAge": "School_Age_High",
        "NurseryProvision (name)": "School_Nursery", 
        "OfficialSixthForm (name)": "School_Sixth_Form",
        "Gender (name)": "School_Gender",
        "ReligiousCharacter (name)": "School_Religious_Character",
        "AdmissionsPolicy (name)": "School_Admissions_Policy",
        "SpecialClasses (name)": "School_Special_Classes",
        "Boarders (name)": "School_Boarders", 
        "SchoolCapacity": "School_Capacity",
        "NumberOfPupils": "School_Number_of_Pupils",
        "PercentageFSM": "School_Percentage_FSM",
        "OpenDate": "School_Open_Date",
        "CloseDate": "School_Close_Date",
        "ReasonEstablishmentOpened (name)": "School_Open_Reason",
        "ReasonEstablishmentClosed (name)": "School_Close_Reason",
        "TrustSchoolFlag (name)": "School_Trust_Flag",
        "Trusts (name)": "School_Trust_Name"
}
print(f"Defined {len(columns_to_keep)} columns to keep")

Defined 31 columns to keep


In [6]:
# DEBUG CODE SNIPPET #

# read in data
tmp = "C:/Users/stefanie.meliss/OneDrive - Ambition Institute/Insights and Data/General/05 Data library/External Data/GIAS/2025-26/edubaseallchildrencentre20251001.csv"
tmp = "C:/Users/stefanie.meliss/OneDrive - Ambition Institute/Insights and Data/General/05 Data library/External Data/GIAS/2025-26/edubasealldata20251001.csv"
df = pd.read_csv(tmp, encoding='latin1', low_memory=False)

# Find which columns are present
available_cols = [col for col in columns_to_keep.keys() if col in df.columns]
print(f"Identified {len(available_cols)} available columns of {len(columns_to_keep)} columns to keep")

# Select and rename available columns
df = df[available_cols].rename(columns={col: columns_to_keep[col] for col in available_cols})

# Add missing columns as NA
for col, new_col in columns_to_keep.items():
    if new_col not in df.columns:
        df[new_col] = pd.NA
df.head()

# Add download date and source file #

# Add source file as column
df["Source_File"] = os.path.basename(tmp)

# Search for an 8-digit number using \d{8} in file name (as extracted from path)
match = re.search(r"(\d{8})", os.path.basename(tmp)) 

# Format date
if match:
    date_str = match.group(1)
    print(f"Extracted date: {date_str}")  # Output: 20211001

    # Convert to a readable date format
    date_obj = datetime.strptime(date_str, "%Y%m%d")
    formatted_date = date_obj.strftime("%d-%m-%Y")
    print(f"Formatted date: {formatted_date}")  # Output: 01-10-2021
else:
    print("No date found in file name.")

# add as column
df["Data_Download_Date"] = pd.to_datetime(formatted_date, format="%d-%m-%Y")

df.head()

Identified 31 available columns of 31 columns to keep
Extracted date: 20251001
Formatted date: 01-10-2025


,Census_Date,URN,School_Name,LA_code,Estab_Number,Local_Authority,Postcode,Region,Urban_Rural,School_Type,School_Type_Group,School_Status,School_Phase,School_Age_Low,School_Age_High,School_Nursery,School_Sixth_Form,School_Gender,School_Religious_Character,School_Admissions_Policy,School_Special_Classes,School_Boarders,School_Capacity,School_Number_of_Pupils,School_Percentage_FSM,School_Open_Date,School_Close_Date,School_Open_Reason,School_Close_Reason,School_Trust_Flag,School_Trust_Name,Source_File,Data_Download_Date
0,16-01-2025,100000,The Aldgate School,201,3614.0,City of London,EC3A 5DE,London,Urban: Nearer to a major town or city,Voluntary aided school,Local authority maintained schools,Open,Primary,3.0,11.0,Has Nursery Classes,Does not have a sixth form,Mixed,Church of England,Not applicable,No Special Classes,No boarders,245.0,249.0,23.3,NaN,NaN,Not applicable,Not applicable,Not applicable,NaN,edubasealldata20251001.csv,2025-10-01
1,18-01-2024,100001,City of London School for Girls,201,6005.0,City of London,EC2Y 8BB,London,Urban: Nearer to a major town or city,Other independent school,Independent schools,Open,Not applicable,10.0,18.0,No Nursery Classes,Has a sixth form,Girls,NaN,Selective,No Special Classes,No boarders,860.0,780.0,NaN,01-01-1920,NaN,Not applicable,Not applicable,Not applicable,NaN,edubasealldata20251001.csv,2025-10-01
2,18-01-2024,100002,St Paul's Cathedral School,201,6006.0,City of London,EC4M 9AD,London,Urban: Nearer to a major town or city,Other independent school,Independent schools,Open,Not applicable,4.0,13.0,No Nursery Classes,Does not have a sixth form,Mixed,Church of England,Not applicable,No Special Classes,Boarding school,285.0,285.0,NaN,01-01-1939,NaN,Not applicable,Not applicable,Not applicable,NaN,edubasealldata20251001.csv,2025-10-01
3,18-01-2024,100003,City of London School,201,6007.0,City of London,EC4V 3AL,London,Urban: Nearer to a major town or city,Other independent school,Independent schools,Open,Not applicable,10.0,18.0,No Nursery Classes,Has a sixth form,Boys,NaN,Not applicable,No Special Classes,No boarders,1100.0,1074.0,NaN,01-01-1919,NaN,Not applicable,Not applicable,Not applicable,NaN,edubasealldata20251001.csv,2025-10-01
4,NaN,100004,Sherborne Nursery School,202,1045.0,Camden,NW5 4LP,London,(England/Wales) Urban major conurbation,Local authority nursery school,Local authority maintained schools,Closed,Nursery,3.0,5.0,Not applicable,Not applicable,Mixed,Does not apply,Not applicable,Not applicable,No boarders,NaN,NaN,NaN,NaN,31-08-1992,Not applicable,Not applicable,Not applicable,NaN,edubasealldata20251001.csv,2025-10-01


In [7]:
# Combine the edubase data for all years

# Read in Edubase data for all schools 
 # edubaseallchildrencentre accounts for children's centres:
  # no census date
  # no information on school type, phase, number pupils etc.
 # edubasealldata accounts for academies, free schools and state funded schools, but not children's centres

# For each file in the edubase data folder, read it in and combine them all into a single dataframe


# List all files in data directory but keep only directories (folders)
all_items = os.listdir(dir_data)
folders = [item for item in all_items if os.path.isdir(os.path.join(dir_data, item))]

# Get full paths for each folder
edubase_folders = [os.path.join(dir_data, folder) for folder in folders]

# Define an empty list to fill with dataframes
df_list = []

# Loop over folders
for folder in edubase_folders:

    logging.info(f"Current folder: {folder}\n")

    # List all items in the folder
    files = os.listdir(folder)

    # Process links #

    # Select links file
    link_file = [f for f in files if f.startswith('links_edubasealldata') and f.endswith('.csv')]
    link_file = [os.path.join(folder, f) for f in link_file]
    
    # Read in URN links
    urn_links = pd.read_csv(link_file[0], encoding ='latin1')

    # Fix data types
    urn_links["URN"] = urn_links["URN"].astype(str)
    urn_links["LinkURN"] = urn_links["LinkURN"].astype(str)

    # Process information about establishments #

    # Select files that start with 'edubaseall' and end with '.csv', but do not start with 'links_'
    target_files = [f for f in files if f.startswith('edubase') and f.endswith('.csv')]

    # join file name with directory path
    csv_files = [os.path.join(folder, target) for target in target_files]

    # loop over csv files
    for file in csv_files:

        logging.info(f"Reading file: {file}")

        # read in data
        df = pd.read_csv(file, encoding='latin1', low_memory=False)
        
        # Find which columns are present
        available_cols = [col for col in columns_to_keep.keys() if col in df.columns]
        logging.info(f"Identified {len(available_cols)} available columns of {len(columns_to_keep)} columns to keep")

        #  Select and rename available columns
        df = df[available_cols].rename(columns={col: columns_to_keep[col] for col in available_cols})

        # Add missing columns as NA
        for col, new_col in columns_to_keep.items():
            if new_col not in df.columns:
                df[new_col] = pd.NA

        # Add download date and source file #

        # Add source file as column
        df["Source_File"] = os.path.basename(file)

        # Search for an 8-digit number using \d{8} in file name (as extracted from path)
        match = re.search(r"(\d{8})", os.path.basename(file)) 

        # Convert to a readable date format
        if match:
            date_str = match.group(1)
            date_obj = datetime.strptime(date_str, "%Y%m%d")
            formatted_date = date_obj.strftime("%d-%m-%Y")
            logging.info(f"Formatted date: {formatted_date}\n")  # Output: 01-10-2021
        else:
            logging.info("No date found in file name.")

        # add as column
        df["Data_Download_Date"] = pd.to_datetime(formatted_date, format="%d-%m-%Y")

        # add academic year
        academic_year = os.path.basename(folder).replace("-", "")
        df[["Academic_Year"]] = academic_year

        # Reorder columns for consistency
        df = df[["Data_Download_Date"] + ["Academic_Year"] + list(columns_to_keep.values()) + ["Source_File"]]# + ['source_file', 'year_folder']]
        
        # Add to df_list
        df_list.append(df)


# Concatenate all DataFrames
edubase_school_info = pd.concat(df_list, ignore_index=True)

2026-08-20 11:38:20,248 INFO Current folder: C:\Users\stefanie.meliss\OneDrive - Ambition Institute\code\DfE_data\data\get-information-about-schools\2020-21

2026-08-20 11:38:20,573 INFO Reading file: C:\Users\stefanie.meliss\OneDrive - Ambition Institute\code\DfE_data\data\get-information-about-schools\2020-21\edubaseallchildrencentre20210730.csv
2026-08-20 11:38:20,737 INFO Identified 9 available columns of 31 columns to keep
2026-08-20 11:38:20,806 INFO Formatted date: 30-07-2021

2026-08-20 11:38:20,826 INFO Reading file: C:\Users\stefanie.meliss\OneDrive - Ambition Institute\code\DfE_data\data\get-information-about-schools\2020-21\edubasealldata20210730.csv
2026-08-20 11:38:24,237 INFO Identified 31 available columns of 31 columns to keep
2026-08-20 11:38:24,274 INFO Formatted date: 30-07-2021

2026-08-20 11:38:24,283 INFO Current folder: C:\Users\stefanie.meliss\OneDrive - Ambition Institute\code\DfE_data\data\get-information-about-schools\2021-22

2026-08-20 11:38:24,390 INFO Re

In [8]:
# Show URN links
# this is only the latest links_edubasealldataYYYYMMDD.csv file as this will contain all previous links
urn_links

,URN,LinkURN,LinkName,LinkType,LinkEstablishedDate
0,100006,134643,CCfL Key Stage 3 PRU,Predecessor - merged,01-01-2022
1,100012,100021,Rhyl Community Primary School,Successor - merged,31-08-2021
2,100016,132245,Kingsgate Primary School,Successor,NaN
3,100017,132245,Kingsgate Primary School,Successor,NaN
4,100021,100012,Carlton Primary School,Predecessor - merged,31-08-2021
...,...,...,...,...,...
35232,402491,401325,Llanfabon Infants School,Predecessor,31-03-2026
35233,402492,400225,Ysgol Y Foryd,Predecessor,31-08-2026
35234,402492,400219,Ysgol Maes Owen,Predecessor,31-08-2026
35235,402493,400220,Ysgol Glan Gele,Predecessor - amalgamated,31-08-2026


In [9]:
# Show results
edubase_school_info

,Data_Download_Date,Academic_Year,Census_Date,URN,School_Name,LA_code,Estab_Number,Local_Authority,Postcode,Region,Urban_Rural,School_Type,School_Type_Group,School_Status,School_Phase,School_Age_Low,School_Age_High,School_Nursery,School_Sixth_Form,School_Gender,School_Religious_Character,School_Admissions_Policy,School_Special_Classes,School_Boarders,School_Capacity,School_Number_of_Pupils,School_Percentage_FSM,School_Open_Date,School_Close_Date,School_Open_Reason,School_Close_Reason,School_Trust_Flag,School_Trust_Name,Source_File
0,2021-07-30,202021,<NA>,20001,The Starship Children's Centre,873,<NA>,Cambridgeshire,CB23 8DY,East of England,<NA>,Children's centre,<NA>,Open,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,18-12-2009,<NA>,<NA>,<NA>,<NA>,<NA>,edubaseallchildrencentre20210730.csv
1,2021-07-30,202021,<NA>,20002,Carlisle South Petteril Bank Upperby SureStart Children's Centre,909,<NA>,Cumbria,CA1 3BX,North West,<NA>,Children's centre,<NA>,Open,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,21-01-2005,<NA>,<NA>,<NA>,<NA>,<NA>,edubaseallchildrencentre20210730.csv
2,2021-07-30,202021,<NA>,20003,Central Sure Start Children's Centre,852,<NA>,Southampton,SO14 0AU,South East,<NA>,Children's centre,<NA>,Open,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,28-03-2006,<NA>,<NA>,<NA>,<NA>,<NA>,edubaseallchildrencentre20210730.csv
3,2021-07-30,202021,<NA>,20004,Ryde Children's Centre,921,<NA>,Isle of Wight,PO33 2JF,South East,<NA>,Children's centre,<NA>,Open,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,19-10-2005,<NA>,<NA>,<NA>,<NA>,<NA>,edubaseallchildrencentre20210730.csv
4,2021-07-30,202021,<NA>,20006,West Allerdale SureStart Children's Centre,909,<NA>,Cumbria,CA15 8HN,North West,<NA>,Children's centre,<NA>,Open,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,28-03-2006,<NA>,<NA>,<NA>,<NA>,<NA>,edubaseallchildrencentre20210730.csv
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
321922,2026-08-20,202526,NaN,402490,Ysgol Gynradd Glyn-coch,674,2389.0,Rhondda Cynon Taf,CF37 3BP,Wales (pseudo),Urban: Nearer to a major town or city,Welsh establishment,Welsh schools,Proposed to open,Not applicable,NaN,NaN,Not applicable,Not applicable,NaN,NaN,NaN,Not applicable,NaN,NaN,NaN,NaN,01-09-2026,NaN,NaN,NaN,Not applicable,NaN,edubasealldata20260820.csv
321923,2026-08-20,202526,NaN,402491,Nelson Primary School,676,2397.0,Caerphilly,CF46 6HL,Wales (pseudo),Urban: Nearer to a major town or city,Welsh establishment,Welsh schools,Open,Not applicable,NaN,NaN,Not applicable,Not applicable,NaN,NaN,NaN,Not applicable,NaN,NaN,NaN,NaN,01-04-2026,NaN,Result of Amalgamation,NaN,Not applicable,NaN,edubasealldata20260820.csv
321924,2026-08-20,202526,NaN,402492,Ysgol Awel y Môr,662,2277.0,Conwy,LL18 5LE,Wales (pseudo),Urban: Further from a major town or city,Welsh establishment,Welsh schools,Proposed to open,Not applicable,NaN,NaN,Not applicable,Not applicable,NaN,NaN,NaN,Not applicable,NaN,NaN,NaN,NaN,01-09-2026,NaN,Result of Amalgamation,NaN,Not applicable,NaN,edubasealldata20260820.csv
321925,2026-08-20,202526,NaN,402493,Ysgol Tan y Gopa,662,2278.0,Conwy,LL22 7NU,Wales (pseudo),Urban: Further from a major town or city,Welsh establishment,Welsh schools,Proposed to open,Not applicable,NaN,NaN,Not applicable,Not applicable,NaN,NaN,NaN,Not applicable,NaN,NaN,NaN,NaN,01-09-2026,NaN,Result of Amalgamation,NaN,Not applicable,NaN,edubasealldata20260820.csv


In [10]:
# Get data types
edubase_school_info.dtypes

Data_Download_Date            datetime64[us]
Academic_Year                            str
Census_Date                           object
URN                                    int64
School_Name                              str
LA_code                                int64
Estab_Number                          object
Local_Authority                          str
Postcode                                 str
Region                                   str
Urban_Rural                           object
School_Type                              str
School_Type_Group                     object
School_Status                            str
School_Phase                          object
School_Age_Low                        object
School_Age_High                       object
School_Nursery                        object
School_Sixth_Form                     object
School_Gender                         object
School_Religious_Character            object
School_Admissions_Policy              object
School_Spe

In [11]:
# Sort out and dtypes and any other data cleaning

# Set datetimes
edubase_school_info["Census_Date"] = pd.to_datetime(edubase_school_info["Census_Date"], format="%d-%m-%Y", errors='coerce')
edubase_school_info["School_Open_Date"] = pd.to_datetime(edubase_school_info["School_Open_Date"], format="%d-%m-%Y", errors='coerce')
edubase_school_info["School_Close_Date"] = pd.to_datetime(edubase_school_info["School_Close_Date"], format="%d-%m-%Y", errors='coerce')

# Set other dtypes
edubase_school_info = edubase_school_info.astype({
    "URN": str,
    "LA_code": 'Int64',
    "Estab_Number": 'Int64',
    "School_Age_Low": 'Int64',
    "School_Age_High": 'Int64',
    "School_Capacity": 'Int64',
    "School_Number_of_Pupils": 'Int64',
    "School_Percentage_FSM": 'Float64'
})

# Convert columns to pandas’ nullable string type
for col in edubase_school_info.columns:
    if edubase_school_info[col].dtype == 'object':
        edubase_school_info[col] = edubase_school_info[col].astype('string')

In [12]:
# Create LAESTAB and DfE_Number
def make_laestab(row):
    if pd.notna(row['LA_code']) and pd.notna(row['Estab_Number']):
        return f"{int(row['LA_code']):03d}{int(row['Estab_Number']):04d}"
    else:
        return pd.NA
        
def make_dfe_number(row):
    if pd.notna(row['LA_code']) and pd.notna(row['Estab_Number']):
        return f"{int(row['LA_code']):03d}/{int(row['Estab_Number']):04d}"
    else:
        return pd.NA

# Apply functions to create new columns
edubase_school_info['LAESTAB'] = edubase_school_info.apply(make_laestab, axis=1).astype('string')
edubase_school_info['DfE_Number'] = edubase_school_info.apply(make_dfe_number, axis=1).astype('string')

# Reorder columns for consistency
edubase_school_info = edubase_school_info[["Data_Download_Date"] + ["Academic_Year"] + ["LAESTAB"] + ["DfE_Number"] + list(columns_to_keep.values()) + ["Source_File"]]

In [13]:
# Check unique values of each column to spot any unstandardised values
for col in edubase_school_info.columns:
    print(col, edubase_school_info[col].unique())

Data_Download_Date <DatetimeArray>
['2021-07-30 00:00:00', '2022-07-30 00:00:00', '2023-07-30 00:00:00', '2024-07-30 00:00:00', '2025-07-30 00:00:00', '2026-08-20 00:00:00']
Length: 6, dtype: datetime64[us]
Academic_Year <ArrowStringArray>
['202021', '202122', '202223', '202324', '202425', '202526']
Length: 6, dtype: str
LAESTAB <ArrowStringArray>
[     <NA>, '2013614', '2016005', '2016006', '2016007', '2021045', '2021048', '2021100', '2021101', '2022019',
 ...
 '6661104', '6691113', '6657019', '6802332', '6747016', '6742389', '6762397', '6622277', '6622278', '6651106']
Length: 44543, dtype: string
DfE_Number <ArrowStringArray>
[      <NA>, '201/3614', '201/6005', '201/6006', '201/6007', '202/1045', '202/1048', '202/1100', '202/1101', '202/2019',
 ...
 '666/1104', '669/1113', '665/7019', '680/2332', '674/7016', '674/2389', '676/2397', '662/2277', '662/2278', '665/1106']
Length: 44543, dtype: string
Census_Date <DatetimeArray>
['NaT', '2021-01-21 00:00:00', '2019-01-17 00:00:00', '2018-

In [14]:
# List of possible unstandardised NA values
unstandardised_na_values = ['', 'NA', 'N/A', 'null', 'None', 'NaN', 'nan', 'none']

# Dictionary to store columns with unstandardised NA values
cols_with_unstandardised_na = {}

for col in edubase_school_info.columns:
    # Only check object (string) columns
    if edubase_school_info[col].dtype == 'object':
        # Count cells with unstandardised NA values
        mask = edubase_school_info[col].isin(unstandardised_na_values)
        count = mask.sum()
        if count > 0:
            cols_with_unstandardised_na[col] = count

print("Columns with unstandardised NA values:", cols_with_unstandardised_na)

# Sort out the <NA> values
if len(cols_with_unstandardised_na) > 0:
    for col in cols_with_unstandardised_na:
        edubase_school_info[col] = edubase_school_info[col].fillna(np.nan)
else:
    print("No columns with unstandardised NA values found.")

Columns with unstandardised NA values: {}
No columns with unstandardised NA values found.


In [15]:
# Assess data

# Count nulls in each column
percent_nulls = (edubase_school_info.isnull().mean() * 100).round(2)

print(percent_nulls)

Data_Download_Date             0.00
Academic_Year                  0.00
LAESTAB                        5.66
DfE_Number                     5.66
Census_Date                   45.19
URN                            0.00
School_Name                    0.00
LA_code                        0.00
Estab_Number                   5.66
Local_Authority                0.00
Postcode                       2.64
Region                         0.00
Urban_Rural                    8.05
School_Type                    0.00
School_Type_Group              5.40
School_Status                  0.00
School_Phase                   5.40
School_Age_Low                12.80
School_Age_High               12.79
School_Nursery                 5.46
School_Sixth_Form              5.42
School_Gender                  8.05
School_Religious_Character    28.48
School_Admissions_Policy      15.21
School_Special_Classes         5.63
School_Boarders                8.35
School_Capacity               28.36
School_Number_of_Pupils     

#### Output

Save the new data

In [17]:
# save csv file
dir_out = home / 'OneDrive - Ambition Institute' / 'code' / 'DfE_data' / 'data'
os.chdir(dir_out)

edubase_school_info.to_csv("data_gias_estab_pa.csv", index=False)